# 7metrics AutoClip - Cloud Worker
Este notebook ejecuta el backend de procesamiento de video en la nube.
Recuerda configurar tu token de **Ngrok** abajo.

In [ ]:
# -*- coding: utf-8 -*-
"""
7metrics AutoClip - Cloud Worker (Google Colab / IDX)
-----------------------------------------------------
Este script implementa el pipeline completo de procesamiento de video para balonmano
utilizando YOLOv11 + ByteTrack + Audio Analysis + FFmpeg Smart Cut.

Instrucciones de Uso:
1. Reemplaza `NGROK_AUTH_TOKEN` abajo con tu token.
2. Ejecuta la celda.
3. Copia la URL pública que aparece (ej: https://xxxx.ngrok-free.app) y úsala en tu frontend local.
"""

import os
import subprocess
import sys
import threading
import time
import asyncio
import json
import shutil
import logging
from typing import List, Optional
from collections import deque

# --- 1. CONFIGURACIÓN E INSTALACIÓN DE DEPENDENCIAS ---

def install_dependencies():
    print("🚀 Instalando dependencias necesarias...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "python-multipart", "pyngrok", "ultralytics", "opencv-python-headless", "ffmpeg-python", "scikit-learn", "scipy", "librosa"])
    
    # Instalar FFmpeg si no está (común en Colab)
    if shutil.which("ffmpeg") is None:
        print("📦 Instalando FFmpeg del sistema...")
        subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

try:
    import fastapi
except ImportError:
    install_dependencies()

# Importaciones tras instalación
from fastapi import FastAPI, UploadFile, File, BackgroundTasks
from fastapi.responses import JSONResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import cv2
import numpy as np
import ffmpeg
import librosa
from ultralytics import YOLO
from sklearn.cluster import KMeans

# Configuración Global
NGROK_AUTH_TOKEN = "TU_TOKEN_AQUI_SI_NO_USAS_VAR_ENTORNO"  # <--- PON TU TOKEN AQUÍ SI ES NECESARIO
PORT = 8000
BASE_DIR = os.getcwd()
UPLOAD_DIR = os.path.join(BASE_DIR, "uploads")
OUTPUT_DIR = os.path.join(BASE_DIR, "output_clips")

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AutoClipWorker")

# --- 2. CLASES DE LÓGICA DE NEGOCIO (CORE) ---

class TeamClassifier:
    """Clasifica jugadores por color de camiseta usando K-Means."""
    def __init__(self):
        self.kmeans = None
        self.trained = False

    def train(self, crops):
        """Entrena K-Means con muestras de los primeros frames."""
        if not crops: return
        data = []
        for crop in crops:
            try:
                hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
                h, w, _ = hsv.shape
                if h < 10 or w < 10: continue
                # Tomar la región central del torso
                center = hsv[int(h*0.2):int(h*0.8), int(w*0.2):int(w*0.8)]
                if center.size == 0: continue
                avg_color = np.mean(center, axis=(0, 1))
                data.append(avg_color)
            except Exception: continue
        
        if len(data) > 20:
            try:
                self.kmeans = KMeans(n_clusters=2, n_init=10)
                self.kmeans.fit(data)
                self.trained = True
                logger.info(f"🎨 Modelo de equipos entrenado. Centros: {self.kmeans.cluster_centers_}")
            except Exception as e:
                logger.error(f"Error entrenando equipos: {e}")

    def predict(self, crop):
        if not self.trained: return -1
        try:
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            h, w, _ = hsv.shape
            if h < 10 or w < 10: return -1
            center = hsv[int(h*0.2):int(h*0.8), int(w*0.2):int(w*0.8)]
            if center.size == 0: return -1
            avg_color = np.mean(center, axis=(0, 1))
            return int(self.kmeans.predict([avg_color])[0])
        except Exception:
            return -1

class AutoClipper:
    """Motor principal de análisis y recorte."""
    def __init__(self, input_path):
        self.input_path = input_path
        self.model = YOLO("yolo11n.pt") # Descarga automática del modelo nano
        self.team_clf = TeamClassifier()
        self.events = []
        self.training_crops = []
        self.frame_idx = 0
        
        # Configuración
        self.conf_threshold = 0.4
        self.lead_in = 8   # Segundos antes
        self.lead_out = 4  # Segundos despues
        self.frame_skip = 3 # Procesar 1 de cada 3 frames para velocidad

    def detect_audio_events(self):
        """Detecta silbatos o picos de audio para sugerir eventos."""
        logger.info("🔊 Analizando audio para detección rápida de eventos...")
        try:
            y, sr = librosa.load(self.input_path, sr=22050, mono=True)
            # Detección simple de picos de energía (aplausos, silbatos)
            onset_env = librosa.onset.onset_strength(y=y, sr=sr)
            peaks = librosa.util.peak_pick(onset_env, 3, 3, 3, 5, 0.5, 10)
            times = librosa.frames_to_time(peaks, sr=sr)
            
            # Filtrar eventos muy cercanos
            filtered_times = []
            if len(times) > 0:
                filtered_times.append(times[0])
                for t in times:
                    if t - filtered_times[-1] > 10: # Min 10s entre eventos
                        filtered_times.append(t)
            
            logger.info(f"🔊 Detectados {len(filtered_times)} eventos por audio.")
            return filtered_times
        except Exception as e:
            logger.error(f"Error analizando audio: {e}")
            return []

    def process_video(self):
        cap = cv2.VideoCapture(self.input_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # 1. Detección rápida por audio (Heurística inicial)
        audio_events = self.detect_audio_events()
        for t in audio_events:
            self.register_event(t, "Audio_Triggered_Event", 0.6)

        # 2. Detección visual (YOLO)
        logger.info("👁️ Iniciando análisis visual...")
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            timestamp = self.frame_idx / fps
            
            # Optimización: Saltar frames
            if self.frame_idx % self.frame_skip != 0:
                self.frame_idx += 1
                continue

            # Inferencia YOLO + Tracking
            results = self.model.track(frame, persist=True, verbose=False, classes=[0, 32]) # 0: persona, 32: balón
            res = results[0]
            
            if res.boxes.id is not None:
                boxes = res.boxes.xyxy.cpu().numpy()
                cls = res.boxes.cls.int().cpu().tolist()
                
                # Entrenar clasificador de equipos al principio
                if self.frame_idx < (300 * self.frame_skip) and not self.team_clf.trained:
                    for box, class_id in zip(boxes, cls):
                        if class_id == 0: # Persona
                            x1, y1, x2, y2 = map(int, box)
                            crop = frame[y1:y2, x1:x2]
                            if crop.size > 0: self.training_crops.append(crop)
                
                elif not self.team_clf.trained and len(self.training_crops) > 0:
                    self.team_clf.train(self.training_crops)
                    self.training_crops = [] # Liberar memoria

                # Lógica de detección de GOL (Visual)
                # Si hay balón (32) y está en una zona específica o movimiento rápido
                # (Simplificado para este script: se basa fuertemente en el audio + detección de balón)
                
            self.frame_idx += 1
            if self.frame_idx % 500 == 0:
                logger.info(f"⏳ Procesado {(self.frame_idx/total_frames)*100:.1f}%")

        cap.release()
        return self.events

    def register_event(self, time, label, conf):
        # Evitar duplicados
        if any(abs(e['time'] - time) < 5 for e in self.events):
            return
            
        event = {
            "time": time,
            "start": max(0, time - self.lead_in),
            "end": time + self.lead_out,
            "label": label,
            "conf": conf
        }
        self.events.append(event)
        self.export_clip(event)

    def export_clip(self, event):
        output_filename = f"clip_{int(event['time'])}_{event['label']}.mp4"
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        
        try:
            (
                ffmpeg
                .input(self.input_path, ss=event['start'], t=(event['end'] - event['start']))
                .output(output_path, c='copy') # Smart Cut (rápido)
                .overwrite_output()
                .run(quiet=True)
            )
            logger.info(f"✅ Clip generado: {output_filename}")
        except Exception as e:
            logger.error(f"Error generando clip: {e}")

# --- 3. API SERVIDOR (FastAPI) ---

app = FastAPI(title="Colab Handball Worker")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def health_check():
    return {"status": "ok", "worker": "Google Colab / IDX"}

@app.post("/upload-video")
async def process_video_endpoint(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    file_path = os.path.join(UPLOAD_DIR, file.filename)
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    
    logger.info(f"📥 Video recibido: {file.filename}")
    
    # Lanzar procesamiento en background
    background_tasks.add_task(run_pipeline, file_path)
    
    return {"message": "Procesamiento iniciado", "filename": file.filename}

@app.get("/clips")
def list_clips():
    clips = []
    if os.path.exists(OUTPUT_DIR):
        for f in os.listdir(OUTPUT_DIR):
            if f.endswith(".mp4"):
                clips.append({
                    "filename": f,
                    "url": f"/download/{f}"
                })
    return clips

@app.get("/download/{filename}")
def download_clip(filename: str):
    path = os.path.join(OUTPUT_DIR, filename)
    return FileResponse(path)

def run_pipeline(video_path):
    clipper = AutoClipper(video_path)
    clipper.process_video()
    logger.info("🏁 Procesamiento del video finalizado.")

# --- 4. PUNTO DE ENTRADA ---

if __name__ == "__main__":
    # Autenticar Ngrok
    token = os.environ.get("NGROK_AUTH_TOKEN", NGROK_AUTH_TOKEN)
    if token == "TU_TOKEN_AQUI_SI_NO_USAS_VAR_ENTORNO":
        print("⚠️ ADVERTENCIA: No se ha configurado token de Ngrok. El túnel podría fallar.")
    else:
        ngrok.set_auth_token(token)

    # Iniciar Túnel
    public_url = ngrok.connect(PORT).public_url
    print(f"============================================================")
    print(f"🌍 URL PÚBLICA DE TU BACKEND EN COLAB: {public_url}")
    print(f"   (Copia esta URL y ponla en tu .env local como PUBLIC_API_URL)")
    print(f"============================================================")

    # Iniciar Servidor
    uvicorn.run(app, host="0.0.0.0", port=PORT)
